In [3]:
!/opt/anaconda/bin/python -m pip install tensorly

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 57.6 MB/s eta 0:00:00a 0:00:01


In [1]:
import hashlib
import numpy as np
import math
import time
import matplotlib.pyplot as plt
import secrets  
from datetime import datetime
from skimage import io
from PIL import Image
import os
from tensorly.decomposition import tucker
import torch.nn as nn
import tensorly as tl
import hashlib
tl.set_backend('numpy')

In [2]:
# Function to display the encrypted image
def display_image(image, title=""):
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class CNN16x16x16(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),  
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),  
            nn.Conv2d(64, 16, kernel_size=1),
            nn.LeakyReLU(0.1)   
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*16*16, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Data transforms
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# ImageFolder datasets
train_dir = 'ham10000/classified_images_train'
val_dir = 'ham10000/classified_images_val'

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = CNN16x16x16(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item() * imgs.size(0)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    val_loss /= len(val_loader.dataset)
    accuracy = correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.4f}")

# Save the trained feature extractor
torch.save(model.features.state_dict(), "cnn_feature_extractor.pth")
print("Saved feature extractor weights to cnn_feature_extractor.pth")


cpu
Epoch [1/10], Train Loss: 0.9932
Epoch [1/10], Val Loss: 0.8984, Val Acc: 0.6763
Epoch [2/10], Train Loss: 0.8534
Epoch [2/10], Val Loss: 0.8501, Val Acc: 0.6938
Epoch [3/10], Train Loss: 0.7945
Epoch [3/10], Val Loss: 0.8470, Val Acc: 0.6893
Epoch [4/10], Train Loss: 0.7531
Epoch [4/10], Val Loss: 0.8144, Val Acc: 0.6953
Epoch [5/10], Train Loss: 0.7172
Epoch [5/10], Val Loss: 0.8354, Val Acc: 0.7137
Epoch [6/10], Train Loss: 0.6960
Epoch [6/10], Val Loss: 0.7679, Val Acc: 0.7192
Epoch [7/10], Train Loss: 0.6880
Epoch [7/10], Val Loss: 0.7778, Val Acc: 0.7222
Epoch [8/10], Train Loss: 0.6648
Epoch [8/10], Val Loss: 0.7765, Val Acc: 0.7237
Epoch [9/10], Train Loss: 0.6429
Epoch [9/10], Val Loss: 0.7808, Val Acc: 0.7097
Epoch [10/10], Train Loss: 0.6254
Epoch [10/10], Val Loss: 0.8207, Val Acc: 0.7157
Saved feature extractor weights to cnn_feature_extractor.pth


In [5]:
import numpy as np
import hashlib
from datetime import datetime
import secrets
from scipy.linalg import svd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os, hmac, hashlib, math
from pathlib import Path
from collections import Counter

class RobustTensorDecomposition:
    """
    Implementation of Robust Tensor Decomposition (RTD) for image key generation
    Based on Algorithm 1 from the provided paper
    """
    
    def __init__(self, convergence_criterion=1e-6, thresholding_scale=0.01, max_iterations=100):
        self.delta = convergence_criterion
        self.beta = thresholding_scale
        self.max_iterations = max_iterations
    
    def hard_threshold(self, T, threshold):
        """
        Hard thresholding operation: H_ζ(T)
        """
        T_thresh = T.copy()
        T_thresh[np.abs(T_thresh) < threshold] = 0
        return T_thresh

    def rank_l_approximation(self, T, rank_l):
        """
        Implements Procedure 1: GradAscent from RTD paper
        Two phases: (1) Deflation to find r eigenpairs, (2) Refinement via gradient ascent
        """
        n1, n2, n3 = T.shape
        assert n1 == n2 == n3, "RTD requires cubic tensors"
        n = n1
        
        # Parameters from paper
        N1 = max(5, int(n ** 1.1))
        N2 = 20
        
        eigenpairs = []
        T_deflated = T.copy()
        
        # ========== PHASE 1: Deflation (lines 2-10) ==========
        for j in range(rank_l):
            best_lambda = 0.0
            best_v = None
            
            # Multiple random initializations
            for i in range(N1):
                # Step 4: Random initialization θ ~ N(0, I_n)
                theta = np.random.randn(n)
                theta = theta / np.linalg.norm(theta)
                
                # Compute T_j(I, I, θ)
                T_I_I_theta = np.zeros((n, n))
                for k in range(n):
                    T_I_I_theta += theta[k] * T_deflated[:, :, k]
                
                # Check if matrix is too small
                T_I_I_theta_norm = np.linalg.norm(T_I_I_theta)
                if T_I_I_theta_norm < 1e-10:
                    continue
                
                # Get top singular vector
                try:
                    U, s, Vt = np.linalg.svd(T_I_I_theta, full_matrices=False)
                    u = U[:, 0]
                except np.linalg.LinAlgError:
                    continue
                
                # Initialize v^(1)_i ← u
                v = u.copy()
                
                # Step 5-8: Power method iterations
                for t in range(N2):
                    # Compute T_j(I, v^(t), v^(t))
                    T_I_v_v = np.zeros(n)
                    for j_idx in range(n):
                        for l_idx in range(n):
                            T_I_v_v += v[j_idx] * v[l_idx] * T_deflated[:, j_idx, l_idx]
                    
                    # Normalize
                    norm_v = np.linalg.norm(T_I_v_v)
                    if norm_v > 1e-12:
                        v = T_I_v_v / norm_v
                    else:
                        break
                    
                    # Compute λ^(t+1) = T_j(v^(t+1), v^(t+1), v^(t+1))
                    lambda_val = 0.0
                    for a in range(n):
                        for b in range(n):
                            for c in range(n):
                                lambda_val += T_deflated[a, b, c] * v[a] * v[b] * v[c]
                
                # Step 9: Pick the best initialization
                if abs(lambda_val) > abs(best_lambda):
                    best_lambda = lambda_val
                    best_v = v.copy()
            
            # Store eigenpair
            if best_v is not None and abs(best_lambda) > 1e-12:
                eigenpairs.append((best_lambda, best_v))
                
                # Step 10: Deflate
                for a in range(n):
                    for b in range(n):
                        for c in range(n):
                            T_deflated[a, b, c] -= best_lambda * best_v[a] * best_v[b] * best_v[c]
        
        # ========== PHASE 2: Gradient Ascent ==========
        T_original = T.copy()
        
        for j in range(len(eigenpairs)):
            lambda_j, v_j = eigenpairs[j]
            v = v_j.copy()
            
            max_grad_iterations = 100
            convergence_tol = 1e-8
            
            for t in range(max_grad_iterations):
                v_old = v.copy()
                
                # Compute T(I, v^(t), v^(t))
                T_I_v_v = np.zeros(n)
                for j_idx in range(n):
                    for l_idx in range(n):
                        T_I_v_v += v[j_idx] * v[l_idx] * T_original[:, j_idx, l_idx]
                
                # Compute λ = T(v, v, v)
                lambda_val = 0.0
                for a in range(n):
                    for b in range(n):
                        for c in range(n):
                            lambda_val += T_original[a, b, c] * v[a] * v[b] * v[c]
                
                # Gradient ascent update
                v_norm_sq = np.dot(v, v)
                step_size = 1.0 / (4 * abs(lambda_val) * (1 + abs(lambda_val) / np.sqrt(n)) + 1e-10)
                gradient_term = T_I_v_v - lambda_val * v_norm_sq * v
                v = v + step_size * gradient_term
                
                # Normalize v
                v_norm = np.linalg.norm(v)
                if v_norm > 1e-12:
                    v = v / v_norm
                else:
                    break
                
                # Check convergence
                relative_change = np.linalg.norm(v - v_old) / (np.linalg.norm(v_old) + 1e-12)
                if relative_change < convergence_tol:
                    break
            
            # Update eigenpair with refined values
            lambda_refined = 0.0
            for a in range(n):
                for b in range(n):
                    for c in range(n):
                        lambda_refined += T_original[a, b, c] * v[a] * v[b] * v[c]
            
            eigenpairs[j] = (lambda_refined, v)
        
        # Step 16: Construct L_hat
        if len(eigenpairs) == 0:
            return np.zeros_like(T)
        
        top_eigenpairs = sorted(eigenpairs, key=lambda x: abs(x[0]), reverse=True)[:rank_l]
        
        # Construct L_hat
        L_hat = np.zeros_like(T)
        for lambda_j, u_j in top_eigenpairs:
            for a in range(n):
                for b in range(n):
                    for c in range(n):
                        L_hat[a, b, c] += lambda_j * u_j[a] * u_j[b] * u_j[c]
        
        return L_hat





    def estimate_eigenvalue(self, T, k):
        """
        Estimate top-k eigenvalues using power method with deflation
        Returns eigenvalues in DESCENDING order
        """
        T = T.astype(np.float64)
        n1, n2, n3 = T.shape
        assert n1 == n2 == n3, "Requires cubic tensor"
        n = n1
        
        N1 = 5  # Random initializations
        N2 = 20  # Power iterations
        
        eigenvalues = []
        T_deflated = T.copy()
        
        for eigen_idx in range(min(k, n)):
            best_lambda = 0.0
            best_v = None
            
            # Multiple random initializations
            for init_idx in range(N1):
                v = np.random.randn(n)
                v = v / np.linalg.norm(v)
                
                # Power method iterations
                for power_iter in range(N2):
                    # Compute T(I, v, v)
                    T_I_v_v = np.zeros(n)
                    for j_idx in range(n):
                        for l_idx in range(n):
                            T_I_v_v += v[j_idx] * v[l_idx] * T_deflated[:, j_idx, l_idx]
                    
                    norm_v = np.linalg.norm(T_I_v_v)
                    if norm_v > 1e-12:
                        v = T_I_v_v / norm_v
                
                # Compute λ = T(v, v, v)
                lambda_val = 0.0
                for a in range(n):
                    for b in range(n):
                        for c in range(n):
                            lambda_val += T_deflated[a, b, c] * v[a] * v[b] * v[c]
                
                if abs(lambda_val) > abs(best_lambda):
                    best_lambda = lambda_val
                    best_v = v.copy()
            
            if best_v is not None and abs(best_lambda) > 1e-12:
                eigenvalues.append(abs(best_lambda))  # Store absolute value
                
                # Deflate: T ← T - λ v⊗v⊗v (use SIGNED lambda)
                for a in range(n):
                    for b in range(n):
                        for c in range(n):
                            T_deflated[a, b, c] -= best_lambda * best_v[a] * best_v[b] * best_v[c]
            else:
                break
        
        return sorted(eigenvalues, reverse=True)  # Return in descending order

    def rtd_decomposition(self, T, target_rank):
        """
        Algorithm 1: RTD (Tensor Robust PCA)
        Follows exact structure from paper
        """
        n1, n2, n3 = T.shape
        assert n1 == n2 == n3, "RTD requires cubic tensors"
        n = n1
        
        
        # Step 2: Initialize
        eigenvals = self.estimate_eigenvalue(T, 1)
        sigma_1_T = eigenvals[0] if eigenvals else np.linalg.norm(T.flatten())
        
        
        zeta_0 = self.beta * sigma_1_T
        
        L = np.zeros_like(T)
        S = self.hard_threshold(T - L, zeta_0)
        
        
        # Step 3: Stage loop l = 1 to r
        for stage_l in range(1, target_rank + 1):
            
            # Step 4: Compute τ
            norm_T_minus_S = np.linalg.norm((T - S).flatten())
            tau = max(1, int(10 * np.log(n) * self.beta * (norm_T_minus_S ** 2) / self.delta))
            tau = min(tau, self.max_iterations)
            
            
            # Inner loop: t = 0 to τ
            for t in range(tau):
                # Step 5: L^(t+1) = P_l(T - S^(t))
                L_new = self.rank_l_approximation(T - S, stage_l)
                
                # Step 7: Update threshold ζ_{t+1}
                residual = T - L_new
                eigenvals_residual = self.estimate_eigenvalue(residual, stage_l + 1)
                
                if len(eigenvals_residual) > stage_l:
                    sigma_l_plus_1 = eigenvals_residual[stage_l]
                    sigma_l = eigenvals_residual[stage_l - 1]
                    zeta = self.beta * (sigma_l_plus_1 + 0.5 * t * sigma_l)
                else:
                    zeta = zeta_0 / (t + 1)
                
                # Step 6: S^(t+1) = H_ζ(T - L^(t+1))
                S_new = self.hard_threshold(residual, zeta)
                
                # CRITICAL FIX: Compute changes BEFORE updating
                L_change = np.linalg.norm((L_new - L).flatten())
                S_change = np.linalg.norm((S_new - S).flatten())
                
                
                # UPDATE L and S FIRST
                L = L_new
                S = S_new
                
                # Check convergence AFTER updating (and skip iteration 0)
                if t > 0 and L_change < self.delta and S_change < self.delta:
                    print(f"  Converged at iteration {t} (ΔL={L_change:.6e}, ΔS={S_change:.6e})")
                    break
            
            # Step 8: Stopping condition check
            eigenvals_L = self.estimate_eigenvalue(L, stage_l + 1)
            if len(eigenvals_L) > stage_l:
                condition = self.beta * eigenvals_L[stage_l]
                threshold = self.delta / (2 * n)
                print(f"  Stopping condition: {condition:.6e} < {threshold:.6e}")
                if condition < threshold:
                    print(f"  Early stopping at stage {stage_l}")
                    break
        
        # Step 9: Return L_hat, S_hat
        print(f"\n RTD Complete ")
        print(f"Final ||L|| = {np.linalg.norm(L.flatten()):.6f}")
        print(f"Final ||S|| = {np.linalg.norm(S.flatten()):.6f}")
        print(f"Final S sparsity: {np.count_nonzero(S)}/{S.size} ({100*np.count_nonzero(S)/S.size:.1f}%)")
        
        return L, S


    


class CNNFeatureExtractor:
    """CNN-based 16x16x16 tensor generator using CNN for 64x64 RGB input."""
    def __init__(self, device='cpu', weights_path=None):
        self.device = device
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1),  
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1), 
            nn.Conv2d(64, 16, kernel_size=1),
            nn.LeakyReLU(0.1)   
        ).to(device)
        
        if weights_path is not None:
            self.model.load_state_dict(torch.load(weights_path, map_location=device))
            print(f"[INFO] Loaded weights from {weights_path}")

        self.model.eval()

        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

    def extract_features(self, image):
        if isinstance(image, np.ndarray):
            if image.max() <= 1.0:
                image = (image * 255).astype(np.uint8)
            image = Image.fromarray(image)
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            features = self.model(img_tensor)
        features_np = features.squeeze(0).cpu().numpy()
        return features_np


def image_to_tensor_cnn(image):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    weights_path = 'cnn_feature_extractor.pth'
    extractor = CNNFeatureExtractor(device=device, weights_path=weights_path)
    tensor = extractor.extract_features(image)
    return tensor





def extract_tensor_features(T, L, S):
    """
    Extract meaningful features from the original tensor (T) and its 
    low-rank (L) and sparse (S) components
    """
    features = {}
    
    # Features from low-rank component L
    features['L_frobenius_norm'] = np.linalg.norm(L.flatten())
    L_reshaped = L.reshape(L.shape[0], -1)
    features['L_spectral_norm'] = np.linalg.norm(L_reshaped, 2)
    features['L_nuclear_norm'] = np.sum(np.linalg.svd(L_reshaped, compute_uv=False))
    features['L_mean'] = np.mean(L)
    features['L_std'] = np.std(L)
    
    # Features from sparse component S
    features['S_l0_norm'] = np.count_nonzero(S)
    features['S_l1_norm'] = np.sum(np.abs(S))
    features['S_max'] = np.max(np.abs(S))
    features['S_mean'] = np.mean(S)
    
    # Combined features
    reconstruction = L + S
    features['reconstruction_error'] = np.linalg.norm((T - reconstruction).flatten()) / np.linalg.norm(T.flatten())
    features['decomposition_ratio'] = features['L_frobenius_norm'] / (features['S_l1_norm'] + 1e-10)
    
    return features


def symmetrize_tensor(T):
    """Symmetrize cubic tensor - required for RTD"""
    n = T.shape[0]
    assert T.shape == (n, n, n), "Must be cubic"
    
    T_sym = np.zeros_like(T)
    for i in range(n):
        for j in range(n):
            for k in range(n):
                T_sym[i,j,k] = (T[i,j,k] + T[i,k,j] + T[j,i,k] + 
                                T[j,k,i] + T[k,i,j] + T[k,j,i]) / 6.0
    return T_sym


def extract_stable_features(L):
    """Extract cryptographically stable features from L"""
    n1, n2, n3 = L.shape
    features = []
    
    for k in range(n3):
        channel_slice = L[:, :, k]
        
        mean_val = np.mean(channel_slice)
        std_val = np.std(channel_slice)
        frobenius_val = np.linalg.norm(channel_slice.flatten())
        max_val = np.max(channel_slice)
        min_val = np.min(channel_slice)
        
        # Normalize using tanh
        mean_norm = (np.tanh(mean_val * 10) + 1) / 2
        std_norm = (np.tanh(std_val * 10) + 1) / 2
        frob_norm = (np.tanh(frobenius_val) + 1) / 2
        max_norm = (np.tanh(max_val * 10) + 1) / 2
        min_norm = (np.tanh(min_val * 10) + 1) / 2
        
        features.extend([mean_norm, std_norm, frob_norm, max_norm, min_norm])
    
    return features

def enhanced_key_generation(image, timestamp, mu, nonce=None, target_rank=2):
    """Enhanced key generation with RTD - Using all 16 L channels"""
    if nonce is None:
        nonce = secrets.token_hex(16)
    
    print(f"\n=== Using CNN-based tensor generation ===")
    tensor = image_to_tensor_cnn(image)  # Returns (16, 16, 16)
    
    # Verify shape
    assert tensor.shape == (16, 16, 16), f"Expected (16,16,16), got {tensor.shape}"
    
    # Symmetrize tensor for RTD
    tensor = symmetrize_tensor(tensor)
    print(f"Tensor symmetrized: {tensor.shape}")

    # Scale tensor
    tensor = tensor * 100
    print(f"Tensor scaled: min={tensor.min():.2f}, max={tensor.max():.2f}, norm={np.linalg.norm(tensor.flatten()):.2f}")
    
    # Apply RTD
    rtd = RobustTensorDecomposition(
        thresholding_scale=0.01,
        convergence_criterion=1e-4,
        max_iterations=100
    )
    L, S = rtd.rtd_decomposition(tensor, target_rank)
    
    # Extract channel-wise sums from all 16 channels 
    n1, n2, n3 = L.shape  # (16, 16, 16)
    channel_sums = []
    
    for k in range(n3):  # Loop through all 16 channels
        channel_sum = np.sum(L[:, :, k])
        channel_sums.append(channel_sum)
        print(f"Channel {k+1} sum: {channel_sum:.6f}")
    
    # Convert channel sums to scaled integers
    channel_sums_scaled = []
    for i, ch_sum in enumerate(channel_sums):
        scaled_sum = int(ch_sum * 1e15) % (10**15)
        channel_sums_scaled.append(scaled_sum)
    
    # Create sum string by concatenating all 16 channel values
    sum_string = ''.join(str(ch_sum) for ch_sum in channel_sums_scaled) + timestamp + nonce
    print(f"Channel-wise concatenated string length: {len(sum_string)}")
    
    # Hash using SHA512
    hashed_sum = hashlib.sha512(sum_string.encode()).hexdigest()
    
    # Divide into 8 parts
    parts = [hashed_sum[i*16:(i+1)*16] for i in range(8)]
    
    keys = []
    psk_file = "../project/psk/psk.key"
    for i, part in enumerate(parts):
        decimal_value = int(part, 16)
        first_15_digits = str(decimal_value)[:15]
        extracted_value = int(first_15_digits) if first_15_digits else 0
        normalized_value = extracted_value / 10**15
        with open(psk_file,"a+") as f:
            f.write(str(normalized_value) + "\n")
        keys.append(normalized_value)
        print(f"RTD Key {i+1}: {normalized_value}")

#    joined = ','.join(f"{f:.12f}" for f in keys)
#    key_bytes = hashlib.sha256(joined.encode()).digest()
#    print(key_bytes.hex())
#    psk_file = '../project/psk/psk.key'
#    with open(psk_file, "a") as f:
#        f.write(key_bytes.hex())
    
    return keys, channel_sums, L, S, tensor



def rtd_key_generation(fake_image, timestamp, mu, nonce=None, target_rank=2):
    """
    Args:
        fake_image: input image
        timestamp: timestamp string
        mu: chaotic map parameter
        nonce: optional nonce
        target_rank: RTD rank
    """
    print(f"\nUsing RTD-based key generation with tensor formation...")
    
    # Generate RTD-based keys
    keys, feature_values, L, S, tensor = enhanced_key_generation(
        fake_image, timestamp, mu, nonce, target_rank)
    
    # Extract feature vector stats
    features = extract_tensor_features(tensor, L, S)
    
    # Print RTD information
    print(f"\nRTD Decomposition Stats:")
    print(f"  - Low-rank component shape: {L.shape}")
    print(f"  - Sparse component non-zeros: {np.count_nonzero(S)}")
    print(f"  - L Frobenius norm: {features['L_frobenius_norm']:.6f}")
    print(f"  - L Spectral norm: {features['L_spectral_norm']:.6f}")
    print(f"  - L Nuclear norm: {features['L_nuclear_norm']:.6f}")
    print(f"  - S L0 norm (sparsity): {features['S_l0_norm']}")
    print(f"  - S L1 norm: {features['S_l1_norm']:.6f}")
    print(f"  - Reconstruction error: {features['reconstruction_error']:.6f}")
    print(f"  - Decomposition ratio: {features['decomposition_ratio']:.6f}")
    
    return keys


In [6]:
# Define the 1D Exponential Chebyshev Map (1-DEC)
def one_dec_map(y, mu):
    # Ensure y stays within [-1, 1] for valid arccos computation
    y = np.clip(y, -1, 1)
    return 1 - 2 * (np.cos(np.arccos(y) * np.exp(abs(mu)) * np.arccos(y)))**2

# Function to generate the nonce
def generate_nonce():
    return secrets.token_hex(16)  # Generate a 16-byte (128-bit) random hex string

# Function to generate chaotic sequence using the 1-DEC map
def chaotic_system(initial_value, mu, iterations=150):
    chaotic_sequence = []
    y = initial_value
    for _ in range(iterations):
        y = one_dec_map(y, mu)
        chaotic_sequence.append(y)
    return chaotic_sequence

# Function to divide hashed sum into 8 parts
def divide_into_eight_parts(hashed_sum):
    # Ensure the hashed_sum is divided into 8 equal parts (128 hex characters / 8 = 16)
    parts = [hashed_sum[i*16:(i+1)*16] for i in range(8)]
    return parts

# Function to convert part to decimal, extract first 15 digits, and normalize
def extract_and_normalize(part):
    # Step 1: Convert hex part to decimal
    decimal_value = int(part, 16)  # Convert the hexadecimal part to decimal    
    # Step 2: Extract the first 15 digits by converting to string and slicing
    first_15_digits = str(decimal_value)[:15]  # Extract the first 15 digits
    
    # Step 3: Convert the first 15 digits back to an integer
    extracted_value = int(first_15_digits)
    
    # Step 4: Normalize by dividing by 10^15
    normalized_value = extracted_value / 10**15
    return normalized_value

In [7]:
# Select 4 decoy images from DCGAN generated images
image_paths = [
    'mri.jpg',
    'AFVAE-CDL-image.png', 
    'chest-x-ray.jpg',
    'baboon.jpg'
]

# Load and resize all 4 images to 64x64
images = []
for i, path in enumerate(image_paths):
    img = Image.open(path)
    print(f"Original image {i+1} size: {img.size}")
    
    # Resize to 64x64 using high-quality resampling
    img_resized = img.resize((64, 64), Image.Resampling.LANCZOS)
    img_array = np.array(img_resized)
    images.append(img_array)
    print(f"Resized image {i+1} shape: {img_array.shape}")

# Convert all images to RGB (3 channels)
for i in range(len(images)):
    if images[i].shape[2] == 4:  # RGBA image
        images[i] = images[i][:, :, :3]  # Remove alpha channel
        print(f"Converted image {i+1} from RGBA to RGB")
    elif images[i].shape[2] != 3:
        if images[i].shape[2] == 1:  # Grayscale
            images[i] = np.repeat(images[i], 3, axis=2)
        else:
            images[i] = images[i][:, :, :3]  # Take first 3 channels
        print(f"Converted image {i+1} to 3 channels")

# Verify all images now have 3 channels
for i, img in enumerate(images):
    print(f"Final image {i+1} shape: {img.shape}")

# Create 2x2 grid (each image is 64x64, so final grid will be 128x128)
print("\nCreating 2x2 grid...")

# Top row: concatenate images[0] and images[1] horizontally
top_row = np.concatenate([images[0], images[1]], axis=1)
print(f"Top row shape: {top_row.shape}")

# Bottom row: concatenate images[2] and images[3] horizontally  
bottom_row = np.concatenate([images[2], images[3]], axis=1)
print(f"Bottom row shape: {bottom_row.shape}")

# Final grid: concatenate top and bottom rows vertically
original_image = np.concatenate([top_row, bottom_row], axis=0)
print(f"Final 2x2 grid shape: {original_image.shape}")


# Current timestamp in the specified format
timestamp = datetime.now().strftime("%d%m%Y%H%M%S")
    
# Chaotic map parameter (mu)
mu = 2
    
# Generate a nonce
nonce = generate_nonce()
    
# Generate 8 keys in a single call
#while True:
keys_original = rtd_key_generation(
        original_image, 
        timestamp, 
        mu, 
        nonce,
        target_rank=2
    )


# Unpack the keys
K1, K2, K3, K4, K5, K6, K7, K8 = keys_original

# Generate chaotic sequences for each key
keys_seq = [np.array(chaotic_system(k, 2, 1500000)) for k in keys_original]
# concatenate all arrays into one large array
#print(keys_seq)
keys_seq_u8 = [
    (np.abs(arr) * 1e15).astype(np.uint64) % 256
    for arr in keys_seq
]
with open("../project/psk/keys_seq_u8.bin", "wb") as f:
    for arr in keys_seq_u8:
        arr.astype(np.uint8).tofile(f)

#with open("../project/psk/chaotic.bin", "wb") as f:
#    f.write(all_values.tobytes())

#with open("../project/psk/chaotic.bin","rb") as f:
#    data = f.read()

#d = hashlib.sha512(data).digest()   # raw 64 bytes
#with open("../project/psk/psk.key","wb") as out:
#    out.write(d)

# convert to bytes efficiently
#key_bytes = hashlib.sha512(all_values.tobytes()).digest()
#print(key_bytes.hex())
#psk_file = '../project/psk/psk.key'
#with open(psk_file, "w") as f:
#    f.write(key_bytes.hex())

Original image 1 size: (2040, 2040)
Resized image 1 shape: (64, 64, 3)
Original image 2 size: (157, 143)
Resized image 2 shape: (64, 64, 4)
Original image 3 size: (526, 400)
Resized image 3 shape: (64, 64, 3)
Original image 4 size: (512, 512)
Resized image 4 shape: (64, 64, 3)
Converted image 2 from RGBA to RGB
Final image 1 shape: (64, 64, 3)
Final image 2 shape: (64, 64, 3)
Final image 3 shape: (64, 64, 3)
Final image 4 shape: (64, 64, 3)

Creating 2x2 grid...
Top row shape: (64, 128, 3)
Bottom row shape: (64, 128, 3)
Final 2x2 grid shape: (128, 128, 3)

Using RTD-based key generation with tensor formation...

=== Using CNN-based tensor generation ===
[INFO] Loaded weights from cnn_feature_extractor.pth
Tensor symmetrized: (16, 16, 16)
Tensor scaled: min=-8.01, max=69.54, norm=963.72
  Converged at iteration 44 (ΔL=4.658476e-05, ΔS=1.525879e-05)
  Stopping condition: 6.779333e-08 < 3.125000e-06
  Early stopping at stage 1

 RTD Complete 
Final ||L|| = 747.595947
Final ||S|| = 63.7488

In [4]:
# Convert the image to red, green, and blue channels; convert channel pixels to bitstream and concatenate
def convert_to_bitstream(image):
    bitstream = np.concatenate([
        np.unpackbits(image[:, :, channel], axis=None) for channel in range(3)
    ])
    return bitstream

# Create sequence S1 from the bitstream, taking 3 bits at a time
def create_sequence(bitstream):
    S1 = bitstream[:len(bitstream) // 3 * 3].reshape(-1, 3)
    return S1

# Create new sequence S2 using XOR between S1 and the previous S2 term
def create_new_sequence_using_xor(S1):
    S2 = np.empty_like(S1)  # Preallocate the array for S2
    S2[0] = S1[0]           # First term remains the same
    np.bitwise_xor.accumulate(S1, out=S2)  # Perform cumulative XOR in bulk
    return S2

# Convert S2 back to RGB image
def convert_to_rgb_image_encrypt(S2, original_shape):
    """
    Convert S2 back to an RGB image while ensuring the correct bitstream size and structure.
    """
    bitstream = S2.flatten().astype(np.uint8)
    num_pixels = original_shape[0] * original_shape[1]

    # Slice the bitstream for each channel
    red_bits, green_bits, blue_bits = np.split(bitstream[:num_pixels * 24], 3)

    # Pack bits into bytes and reshape to original channel dimensions
    red_channel = np.packbits(red_bits).reshape(original_shape[:2])
    green_channel = np.packbits(green_bits).reshape(original_shape[:2])
    blue_channel = np.packbits(blue_bits).reshape(original_shape[:2])

    # Stack channels into the final image
    return np.stack((red_channel, green_channel, blue_channel), axis=-1)


# Intershuffling step
def intershuffling(I, K1, K2, K3):
    # Get the shape of the input image
    s = I.shape
    K1 = K1[:s[0]]
    K2 = K2[:s[1]]
    K3 = K3[:s[2]]
    K1 = abs(K1) * 10**15
    K2 = abs(K2) * 10**15
    K3 = abs(K3) * 10**15
    K1 = np.array(list(map(int, K1)))
    K2 = np.array(list(map(int, K2)))
    K3 = np.array(list(map(int, K3)))
    # print(K1,K2,K3)
    # print(K2[s[1]-1])
    # Create a copy of the image to avoid modifying the original during shuffling
    shuffled_image = np.copy(I)

    # Iterate over the dimensions of the image and apply shuffling
    for i in range(s[0]):
        for j in range(s[1]):
            for k in range(s[2]):
                # Calculate new indices based on K1, K2, K3
                # print((K3[k]%3)%3)
                new_i = (K1[i]%256)% s[0]
                new_j = (K2[j]%256) % s[1] 
                new_k = (K3[k]%3) % s[2]
                # print(new_i, new_j, new_k)
                # Swap the pixel at (i, j, k) with the calculated new position
                temp = shuffled_image[i, j, k]
                shuffled_image[i, j, k] = shuffled_image[new_i, new_j, new_k]
                shuffled_image[new_i, new_j, new_k] = temp
                # print("one:", i,j, k, "two:", new_i, new_j, new_k)
    return shuffled_image

# def zigzag_xor(image, K4):
#     # # K4 = abs(K4) * 10**15
#     # # K4 = np.array(list(map(int, K4)))
#     # # K4 = K4%256
#     # K4 = int(abs(K4) * 10**15) % 256
#     # result_image = np.copy(image)
#     # for i in range(image.shape[0]):
#     #     for j in range(image.shape[1]):
#     #         for k in range(image.shape[2]):
#     #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4)
#     # return result_image
#     # Compute K4 as an integer modulo 256
#     K4 = int(abs(K4) * 10**15) % 256

#     # Perform XOR operation across the entire RGB image
#     result_image = np.bitwise_xor(image, K4)
#     return result_image

def zigzag_xor(image, K4):
    s = image.shape
    K4 = abs(K4) * 10**15
    K4 = np.array(list(map(int, K4)))
    K4 = K4%256
    # # K4 = int(abs(K4) * 10**15) % 256
    # result_image = np.copy(image)
    # for i in range(image.shape[0]):
    #     for j in range(image.shape[1]):
    #         for k in range(image.shape[2]):
    #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4[i*s[1]*s[2]+j*s[2]+k])
    # return result_image

    # # Calculate chaotic sequence K4 values modulo 256
    # s = image.shape
    # K4 = (np.abs(K4) * 10**15).astype(np.uint64) % 256
    
    # # Flatten K4 and ensure it matches the total number of image elements
    # K4 = np.resize(K4, image.size)  # Reshape or tile K4 to match the image size

    # # Flatten the image, apply XOR, and reshape back to the original shape
    # result_image = np.bitwise_xor(image.flatten(), K4).reshape(s)

    # return result_image

     # Calculate chaotic sequence K4 values modulo 256
    # K4 = (np.abs(K4) * 10**15).astype(np.int64) % 256

    # Flatten K4 and match it to the total number of image elements
    K4 = np.resize(K4, image.size).astype(np.uint8)

    # Create a copy of the original image
    result_image = np.copy(image)

    # Flatten the image for efficient processing
    flat_image = result_image.flatten()

    # Perform XOR operation
    flat_result = np.bitwise_xor(flat_image, K4)

    # Reshape the result back to the original image shape
    result_image = flat_result.reshape(s).astype(np.uint8)

    return result_image


def planet(I, K1):
    # Get the size of the input image
    ImageSize = I.shape
    # Extract the red, green, and blue channels
    RedChannel = I[:, :, 0]
    GreenChannel = I[:, :, 1]
    BlueChannel = I[:, :, 2]
    
    # Reshape the channels into 1D bitstreams
    Redbitstream = RedChannel.flatten()
    Greenbitstream = GreenChannel.flatten()
    Bluebitstream = BlueChannel.flatten()

    # Concatenate the bitstreams from all channels
    TotalBits = np.concatenate((Redbitstream, Greenbitstream, Bluebitstream))
    
    # Initialize an empty array for the mixed bitstream
    Mixedbitstream = np.zeros_like(Redbitstream)

    # Initialize an empty array for the mixed bitstream
    Mixedbitstream = np.zeros_like(TotalBits)

    # K1 = np.array(chaotic_system(K1, 2, len(TotalBits)))
    
    # Scale K1 by 256 and take the absolute value
    K1 = abs(K1) * 256
    # Iterate through the image size
    for i in range(len(TotalBits)):
        if i % 3 == 0:
            # XOR with the red bitstream
            Mixedbitstream[i] = np.bitwise_xor(Redbitstream[i // 3], np.uint8(K1[i]))
        elif i % 3 == 1:
            # XOR with the green bitstream
            Mixedbitstream[i] = np.bitwise_xor(Greenbitstream[i // 3], np.uint8(K1[i]))
        else:
            # XOR with the blue bitstream
            Mixedbitstream[i] = np.bitwise_xor(Bluebitstream[i // 3], np.uint8(K1[i]))
    
    # Reshape the mixed bitstream back into the image size
    RestoredImage = Mixedbitstream.reshape(ImageSize[0], ImageSize[1], ImageSize[2])
    
    return RestoredImage



# Complete encryption process
def encrypt_image(image, key):
    # Step 2: Perform VPD

    K1, K2, K3, K4, K5, K6, K7, K8 = key
    start = time.time()
    
    bitstream = convert_to_bitstream(image)
    S1 = create_sequence(bitstream)
    S2 = create_new_sequence_using_xor(S1)
    I1 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time1: {end-start:.6f} seconds")
    # display_image(I1, "I1")
    # Step 3: Intershuffling using K1, K2, K3
    start = time.time()
    I2 = intershuffling(I1, K1, K2, K3)
    end = time.time()
    print(f"Encryption Time2: {end-start:.6f} seconds")
    # display_image(I2, "I2")
    # Step 4: Zigzag XORing using K4
    start = time.time()
    I3 = zigzag_xor(I2, K4)
    end = time.time()
    print(f"Encryption Time3: {end-start:.6f} seconds")
    # display_image(I3, "I3")
    # Step 5: Perform VPD again
    start = time.time()
    bitstream = convert_to_bitstream(I3)
    S1 = create_sequence(bitstream)
    S2 = create_new_sequence_using_xor(S1)
    I4 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time4: {end-start:.6f} seconds")
    start = time.time()
    
    # display_image(I4, "I4")
    # Step 6: Intershuffling using K5, K6, K7
    I5 = intershuffling(I4, K5, K6, K7)
    end = time.time()
    print(f"Encryption Time5: {end-start:.6f} seconds")
    start = time.time()
    # display_image(I5, "I5")
    # Step 7: Zigzag XORing using K4 again
    I6 = zigzag_xor(I5, K4)
    end = time.time()
    print(f"Encryption Time6: {end-start:.6f} seconds")
    start = time.time()

    # display_image(I6, "I6")
    # Step 8: Final VPD
    bitstream = convert_to_bitstream(I6)
    bits = bitstream
    S1 = create_sequence(bitstream)
    # print("I6 S1:", S1)
    S2 = create_new_sequence_using_xor(S1)
    # print("ENCRYPT S2 BITSTREAM:", len(S2))
    # print("ENCRYPT S2 BITSTREAM:", S2)
    I7 = convert_to_rgb_image_encrypt(S2, image.shape)
    end = time.time()
    print(f"Encryption Time7: {end-start:.6f} seconds")
    start = time.time()
    # display_image(I7, title="I7")
    # Step 9: Planet encryption using K8
    encrypted_image = planet(I7, K8)
    end = time.time()
    print(f"Encryption Time8: {end-start:.6f} seconds")
    
    return encrypted_image

In [5]:
#TODO: Optimize the functions here used in decryption to reduce decryption time

def PlanetDec(C, K8):
    # Get image size and reshape the encrypted image into a flat bitstream
    image_size = C.shape
    mixed_bitstream = C.flatten()
    K8=K8[:len(mixed_bitstream)]
    # K8 = np.array(chaotic_system(K8, 2, len(mixed_bitstream)))
    K8 = abs(K8) * 256
    K8 = K8.astype(np.uint8)

    # Initialize bitstreams for red, green, and blue channels
    red_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    green_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    blue_bitstream = np.zeros(len(mixed_bitstream) // 3, dtype=np.uint8)
    # Reverse the XOR operation with K8
    for i in range(len(mixed_bitstream)):
        if i % 3 == 0:
            red_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i]) #CHANGE THIS LATER
        elif i % 3 == 1:
            green_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i])
        else:
            blue_bitstream[i // 3] = np.bitwise_xor(mixed_bitstream[i], K8[i])
    # Reshape the bitstreams into their original color channel shapes
    red_channel = red_bitstream.reshape(image_size[0], image_size[1])
    green_channel = green_bitstream.reshape(image_size[0], image_size[1])
    blue_channel = blue_bitstream.reshape(image_size[0], image_size[1])

    # Combine the color channels into an RGB image
    restored_image = np.zeros(image_size, dtype=np.uint8)
    restored_image[:, :, 0] = red_channel
    restored_image[:, :, 1] = green_channel
    restored_image[:, :, 2] = blue_channel

    return restored_image

# Step 2: Convert image to bitstream
def ConvertToBitstream(I):
    image = np.copy(I)
    red_channel = np.unpackbits(image[:, :, 0], axis=1)
    green_channel = np.unpackbits(image[:, :, 1], axis=1)
    blue_channel = np.unpackbits(image[:, :, 2], axis=1)
    bitstream = np.concatenate((red_channel.flatten(), green_channel.flatten(), blue_channel.flatten()))
    return bitstream

# Step 3: Create sequence S1
def CreateSequence(bitstream):
    S1 = []
    for i in range(0, len(bitstream), 3):
        S1.append(bitstream[i:i+3])  # Group bits into sequences of 3
    return np.array(S1)

# Step 4: Reverse the XOR sequence (generate S2)
def ReverseSequenceUsingXOR(S1):
    S2 = np.zeros_like(S1)
    S2[0] = S1[0]  # Start with the first element of S1 (this might need to match the encryption more closely)
    for i in range(1, len(S1)):
        S2[i] = np.bitwise_xor(S1[i], S1[i - 1])  # Reverse XOR operation, ensure this mirrors encryption
    return np.array(S2)

def ConvertToRGBImage(S2, image_shape):
    bitstream = np.concatenate(S2).astype(np.uint8)
    
    # Calculate the number of bits per channel (8 bits per pixel)
    num_pixels = image_shape[0] * image_shape[1]
    
    red_bits = bitstream[:num_pixels * 8]
    green_bits = bitstream[num_pixels * 8:2 * num_pixels * 8]
    blue_bits = bitstream[2 * num_pixels * 8:]
    red_channel = np.packbits(red_bits).reshape((image_shape[0], image_shape[1]))
    green_channel = np.packbits(green_bits).reshape((image_shape[0], image_shape[1]))
    blue_channel = np.packbits(blue_bits).reshape((image_shape[0], image_shape[1]))
    # Return the stacked image without the extra dimension
    return np.stack((red_channel, green_channel, blue_channel), axis=-1)


# Step 6: Reverse Zigzag XOR
def ReverseZigzagXOR(image, K4):

    s = image.shape
    K4 = abs(K4) * 10**15
    K4 = np.array(list(map(int, K4)))
    K4 = K4%256
    # # K4 = int(abs(K4) * 10**15) % 256
    # result_image = np.copy(image)
    # for i in range(image.shape[0]):
    #     for j in range(image.shape[1]):
    #         for k in range(image.shape[2]):
    #             result_image[i, j, k] = np.bitwise_xor(image[i, j, k], K4[i*s[1]*s[2]+j*s[2]+k])
    # return result_image

    # Flatten K4 and match it to the total number of image elements
    K4 = np.resize(K4, image.size).astype(np.uint8)

    # Create a copy of the original image
    result_image = np.copy(image)

    # Flatten the image for efficient processing
    flat_image = result_image.flatten()

    # Perform XOR operation
    flat_result = np.bitwise_xor(flat_image, K4)

    # Reshape the result back to the original image shape
    result_image = flat_result.reshape(s).astype(np.uint8)

    return result_image


# Step 7: Reverse Intershuffling
def ReverseInterShuffling(I, K1, K2, K3):
    # Get the shape of the input image
    s = I.shape
    # K1 = np.array(chaotic_system(K1, 2, s[0]))
    # K2 = np.array(chaotic_system(K2, 2, s[1]))
    # K3 = np.array(chaotic_system(K3, 2, s[2]))
    # print(k1,k2,k3)
    K1 = K1[:s[0]]
    K2 = K2[:s[1]]
    K3 = K3[:s[2]]
    K1 = abs(K1) * 10**15
    K2 = abs(K2) * 10**15
    K3 = abs(K3) * 10**15
    K1 = np.array(list(map(int, K1)))
    K2 = np.array(list(map(int, K2)))
    K3 = np.array(list(map(int, K3)))
    
    
    # print(K2[s[1]-1])
    # Create a copy of the image to avoid modifying the original during reverse shuffling
    reverse_shuffled_image = np.copy(I)
    
    # Iterate over the dimensions of the image in reverse order to reverse the shuffling process
    for i in range(s[0]-1, -1, -1):
        for j in range(s[1]-1, -1, -1):
            for k in range(s[2]-1, -1, -1):
                orig_i = (K1[i] % 256) % s[0]
                orig_j = (K2[j] % 256) % s[1]
                orig_k = (K3[k] % 3) % s[2]
                
                # Swap back to the original position
                temp = reverse_shuffled_image[i, j, k]
                reverse_shuffled_image[i, j, k] = reverse_shuffled_image[orig_i, orig_j, orig_k]
                reverse_shuffled_image[orig_i, orig_j, orig_k] = temp
                # print("one:", i,j, k, "two:",  orig_i, orig_j, orig_k)
                

    return reverse_shuffled_image

# Step 8: Decrypt Image (Main Function)
def decrypt_image(encrypted_image, key):
    K1, K2, K3, K4, K5, K6, K7, K8 = key
    image_shape = encrypted_image.shape[:2]  # Extract the original image dimensions
    # Step 1: Reverse planet encryption with K8
    I7 = PlanetDec(encrypted_image, K8)

    # display_image(I7, "Decrypted I7")
    # Step 2: Reverse VPD
    bitstream = ConvertToBitstream(I7)   
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    I6 = ConvertToRGBImage(S2, image_shape)

    # display_image(I6, "Decrypted I6")
    # Step 3: Reverse Zigzag XORing using K4
    I5 = ReverseZigzagXOR(I6, K4)
    # display_image(I5, "decrypted I5")
    # Step 4: Reverse Intershuffling with K5, K6, K7
    I4 = ReverseInterShuffling(I5, K5, K6, K7)
    # display_image(I4, "decrypted I4")
    # Step 5: Reverse VPD again
    bitstream = ConvertToBitstream(I4)
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    I3 = ConvertToRGBImage(S2, image_shape)
    # display_image(I3, "decrypted I3")
    # Step 6: Reverse Zigzag XORing using K4 again
    I2 = ReverseZigzagXOR(I3, K4)
    # display_image(I2, "decrypted I2")
    # Step 7: Reverse Intershuffling with K1, K2, K3
    I1 = ReverseInterShuffling(I2, K1, K2, K3)
    # display_image(I1, "decrypted I1")
    # Step 8: Reverse VPD for the final time
    bitstream = ConvertToBitstream(I1)
    S1 = CreateSequence(bitstream)
    S2 = ReverseSequenceUsingXOR(S1)
    Decrypted_Image = ConvertToRGBImage(S2, image_shape)
    end_time = time.time()
    return Decrypted_Image

In [6]:
# Now encrypt the 2x2 grid image using the generated keys
encrypted_image = encrypt_image(original_image, keys_seq)

# Display the original 2x2 grid image
display_image(original_image, "Original 2x2 Grid Image")

# Display the encrypted image
display_image(encrypted_image, "Encrypted Image")

# Decrypt the image using the same keys
decrypted_image = decrypt_image(encrypted_image, keys_seq)

# Display the decrypted image
display_image(decrypted_image, "Decrypted Image")

NameError: name 'original_image' is not defined